# q03：改良VAD手法の実装

基礎レベル3では，ベースラインの弱点を踏まえ，VAD手法を改良する．

ここでは，単純な全帯域エネルギーではなく，音声帯域のエネルギーとゼロ交差率を利用する．  
さらに，短すぎる音声区間を除去し，中央値フィルタとhangoverで時間方向のばらつきを抑える．

In [ ]:
import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import json



from pathlib import Path

ROOT = Path("./data")
INPUT_DIR = ROOT / "inputs"
ANNOTATION_DIR = ROOT / "annotations"
RESULT_DIR = ROOT / "results"

ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_IDS = [f"A{i}" for i in range(1, 7)]
TEST_IDS = [f"A{i}" for i in range(7, 10)]
ALL_IDS = TRAIN_IDS + TEST_IDS

FRAME_SEC = 0.010
TARGET_FS = 16000

def wav_path(utt_id: str) -> Path:
    return INPUT_DIR / f"{utt_id}.wav"

def interval_path(utt_id: str) -> Path:
    return ANNOTATION_DIR / f"{utt_id}_intervals.csv"

def label_path(utt_id: str) -> Path:
    return ANNOTATION_DIR / f"{utt_id}_labels.csv"



import numpy as np
import pandas as pd
import soundfile as sf

EPS = 1e-12

def load_audio(path, target_fs: int = 16000):
    """WAVを読み込み，モノラルfloat32波形として返す．"""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"音声ファイルが見つかりません: {path}")
    x, fs = sf.read(path, dtype="float32")
    if x.ndim == 2:
        x = x.mean(axis=1)
    if fs != target_fs:
        raise ValueError(f"{path} のサンプリング周波数が {fs} Hz です．{target_fs} Hzで収録した音声を使用してください．")
    return x.astype(np.float32), fs

def frame_signal(x: np.ndarray, fs: int, frame_sec: float = 0.010):
    """10 msごとに非重複フレーム化する．不足分は0で埋める．"""
    frame_len = int(round(frame_sec * fs))
    n_frames = int(np.ceil(len(x) / frame_len))
    pad_len = n_frames * frame_len - len(x)
    if pad_len > 0:
        x = np.pad(x, (0, pad_len))
    frames = x.reshape(n_frames, frame_len)
    times = np.arange(n_frames) * frame_sec
    return frames, times

def hann_window(n: int):
    return np.hanning(n).astype(np.float32)

def short_time_log_energy(x: np.ndarray, fs: int, frame_sec: float = 0.010, use_hann: bool = True):
    frames, times = frame_signal(x, fs, frame_sec)
    if use_hann:
        frames = frames * hann_window(frames.shape[1])[None, :]
    energy = np.mean(frames ** 2, axis=1)
    log_energy = 10.0 * np.log10(EPS + energy)
    return log_energy, times

def relative_energy(log_energy: np.ndarray, percentile: float = 20.0):
    noise_floor = np.percentile(log_energy, percentile)
    score = log_energy - noise_floor
    return score, noise_floor

def apply_hangover(pred: np.ndarray, hangover_frames: int):
    """1から0へ切り替わった直後の指定フレーム数を1にする．"""
    pred = np.asarray(pred, dtype=np.int64).copy()
    if hangover_frames <= 0:
        return pred
    out = pred.copy()
    n = len(pred)
    for m in range(n - 1):
        if pred[m] == 1 and pred[m + 1] == 0:
            end = min(n, m + 1 + hangover_frames)
            out[m + 1:end] = 1
    return out

def labels_from_intervals(intervals, n_frames: int, frame_sec: float = 0.010):
    """[(start_sec, end_sec), ...]を10 msフレームラベルへ変換する．"""
    labels = np.zeros(n_frames, dtype=np.int64)
    for start_sec, end_sec in intervals:
        start = max(0, int(np.floor(start_sec / frame_sec)))
        end = min(n_frames, int(np.ceil(end_sec / frame_sec)))
        labels[start:end] = 1
    return labels

def save_labels_from_intervals(utt_id: str, intervals, duration_sec: float):
    """区間アノテーションCSVとフレームラベルCSVを保存する．"""
    n_frames = int(np.ceil(duration_sec / FRAME_SEC))
    interval_df = pd.DataFrame(intervals, columns=["start_sec", "end_sec"])
    interval_df["label"] = 1
    interval_df.to_csv(interval_path(utt_id), index=False)

    labels = labels_from_intervals(intervals, n_frames, FRAME_SEC)
    label_df = pd.DataFrame({
        "frame": np.arange(n_frames),
        "start_sec": np.arange(n_frames) * FRAME_SEC,
        "end_sec": (np.arange(n_frames) + 1) * FRAME_SEC,
        "label": labels
    })
    label_df.to_csv(label_path(utt_id), index=False)
    return interval_df, label_df

def load_frame_labels(utt_id: str, n_frames: int):
    """q01で保存したフレームラベルを読み込む．長さが合わない場合は切り詰めまたは0埋めする．"""
    path = label_path(utt_id)
    if not path.exists():
        raise FileNotFoundError(f"アノテーションが見つかりません: {path}")
    labels = pd.read_csv(path)["label"].to_numpy(dtype=np.int64)
    if len(labels) < n_frames:
        labels = np.pad(labels, (0, n_frames - len(labels)))
    elif len(labels) > n_frames:
        labels = labels[:n_frames]
    return labels

def confusion_binary(y_true: np.ndarray, y_pred: np.ndarray, positive_label: int = 1):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == positive_label) & (y_pred == positive_label)))
    fp = int(np.sum((y_true != positive_label) & (y_pred == positive_label)))
    fn = int(np.sum((y_true == positive_label) & (y_pred != positive_label)))
    tn = int(np.sum((y_true != positive_label) & (y_pred != positive_label)))
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn}

def f1_from_counts(tp: int, fp: int, fn: int):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1, precision, recall

def macro_f1_score(y_true: np.ndarray, y_pred: np.ndarray):
    """音声クラスと非音声クラスのF1平均を返す．"""
    c_s = confusion_binary(y_true, y_pred, positive_label=1)
    f1_s, p_s, r_s = f1_from_counts(c_s["TP"], c_s["FP"], c_s["FN"])

    c_n = confusion_binary(1 - y_true, 1 - y_pred, positive_label=1)
    f1_n, p_n, r_n = f1_from_counts(c_n["TP"], c_n["FP"], c_n["FN"])

    return {
        "macro_f1": (f1_s + f1_n) / 2,
        "f1_speech": f1_s,
        "precision_speech": p_s,
        "recall_speech": r_s,
        "f1_non_speech": f1_n,
        "precision_non_speech": p_n,
        "recall_non_speech": r_n,
        **c_s
    }

def load_dataset(utt_ids):
    """音声，エネルギー，正解ラベルをIDごとに読み込む．"""
    data = {}
    for utt_id in utt_ids:
        x, fs = load_audio(wav_path(utt_id), TARGET_FS)
        E, times = short_time_log_energy(x, fs, FRAME_SEC, use_hann=True)
        y = load_frame_labels(utt_id, len(E))
        data[utt_id] = {"x": x, "fs": fs, "E": E, "times": times, "y": y}
    return data

def concat_labels_and_preds(y_list, pred_list):
    return np.concatenate(y_list), np.concatenate(pred_list)

## 1．特徴量の定義

改良手法では，次の特徴量を利用する．

- 音声帯域エネルギー：300 Hzから3400 Hzの周波数成分の対数エネルギー  
- ゼロ交差率：フレーム内で波形の符号が変化する割合

In [ ]:
def speech_band_log_energy(x: np.ndarray, fs: int, frame_sec: float = 0.010,
                           low_hz: float = 300.0, high_hz: float = 3400.0,
                           use_hann: bool = True):
    frames, times = frame_signal(x, fs, frame_sec)
    if use_hann:
        frames = frames * hann_window(frames.shape[1])[None, :]

    spec = np.fft.rfft(frames, axis=1)
    freqs = np.fft.rfftfreq(frames.shape[1], d=1.0 / fs)
    band = (freqs >= low_hz) & (freqs <= high_hz)

    power = np.mean(np.abs(spec[:, band]) ** 2, axis=1)
    band_energy = 10.0 * np.log10(EPS + power)
    return band_energy, times

def zero_crossing_rate(x: np.ndarray, fs: int, frame_sec: float = 0.010):
    frames, times = frame_signal(x, fs, frame_sec)
    signs = np.signbit(frames)
    zcr = np.mean(signs[:, 1:] != signs[:, :-1], axis=1)
    return zcr, times

def median_filter_binary(pred: np.ndarray, width: int):
    pred = np.asarray(pred, dtype=np.int64)
    if width <= 1:
        return pred.copy()
    if width % 2 == 0:
        raise ValueError("widthは奇数にしてください")
    pad = width // 2
    padded = np.pad(pred, (pad, pad), mode="edge")
    out = np.zeros_like(pred)
    for i in range(len(pred)):
        out[i] = 1 if np.median(padded[i:i + width]) >= 0.5 else 0
    return out

def remove_short_speech_segments(pred: np.ndarray, min_speech_frames: int):
    pred = np.asarray(pred, dtype=np.int64).copy()
    if min_speech_frames <= 1:
        return pred

    n = len(pred)
    i = 0
    while i < n:
        if pred[i] == 0:
            i += 1
            continue
        start = i
        while i < n and pred[i] == 1:
            i += 1
        end = i
        if end - start < min_speech_frames:
            pred[start:end] = 0
    return pred

def extract_improved_features(x: np.ndarray, fs: int):
    band_E, times = speech_band_log_energy(x, fs, FRAME_SEC)
    S_band, band_noise = relative_energy(band_E, percentile=20.0)
    zcr, _ = zero_crossing_rate(x, fs, FRAME_SEC)
    return {
        "band_E": band_E,
        "S_band": S_band,
        "band_noise": band_noise,
        "zcr": zcr,
        "times": times
    }

def predict_improved_from_features(features, theta: float, zcr_min: float, zcr_max: float,
                                   median_width: int, min_speech_frames: int, Lh: int):
    pred = (features["S_band"] >= theta).astype(np.int64)

    # 極端に低いZCRまたは高いZCRのフレームは，無音や雑音の可能性があるため非音声に寄せる
    zcr = features["zcr"]
    pred = pred & (zcr >= zcr_min) & (zcr <= zcr_max)
    pred = pred.astype(np.int64)

    pred = median_filter_binary(pred, median_width)
    pred = remove_short_speech_segments(pred, min_speech_frames)
    pred = apply_hangover(pred, Lh)
    return pred

## 2．trainデータの特徴量抽出

In [ ]:
train_data = {}
for uid in TRAIN_IDS:
    x, fs = load_audio(wav_path(uid), TARGET_FS)
    features = extract_improved_features(x, fs)
    y = load_frame_labels(uid, len(features["S_band"]))
    train_data[uid] = {"x": x, "fs": fs, "y": y, **features}

for uid, d in train_data.items():
    print(uid, "frames:", len(d["S_band"]), "speech frames:", int(d["y"].sum()))

## 3．改良手法のパラメータ探索

trainデータのmacro-F1が最大となる組み合わせを探索する．  
探索対象は，音声帯域エネルギーしきい値，ZCR範囲，中央値フィルタ幅，最小音声長，hangover長である．

In [ ]:
def evaluate_improved(data, params):
    y_all = []
    pred_all = []
    for uid, d in data.items():
        features = {
            "S_band": d["S_band"],
            "zcr": d["zcr"]
        }
        pred = predict_improved_from_features(features, **params)
        y_all.append(d["y"])
        pred_all.append(pred)
    y_all, pred_all = concat_labels_and_preds(y_all, pred_all)
    return macro_f1_score(y_all, pred_all)

# thetaは0.5 dB刻み
all_scores = np.concatenate([d["S_band"] for d in train_data.values()])
theta_max = max(0.0, np.ceil(np.max(all_scores) * 2) / 2)
theta_grid = np.arange(0.0, theta_max + 0.5, 0.5)

param_results = []
for theta in theta_grid:
    for zcr_min in [0.00, 0.005, 0.010, 0.020]:
        for zcr_max in [0.15, 0.20, 0.25, 0.30, 1.00]:
            if zcr_min >= zcr_max:
                continue
            for median_width in [1, 3, 5]:
                for min_speech_frames in [1, 3, 5]:
                    for Lh in [0, 3, 5, 10, 20]:
                        params = {
                            "theta": float(theta),
                            "zcr_min": float(zcr_min),
                            "zcr_max": float(zcr_max),
                            "median_width": int(median_width),
                            "min_speech_frames": int(min_speech_frames),
                            "Lh": int(Lh)
                        }
                        metrics = evaluate_improved(train_data, params)
                        param_results.append({**params, **metrics})

improved_result_df = pd.DataFrame(param_results)

# 同点なら，より単純な条件を優先する
best_improved = (
    improved_result_df
    .sort_values(
        ["macro_f1", "Lh", "median_width", "min_speech_frames"],
        ascending=[False, True, True, True]
    )
    .iloc[0]
    .to_dict()
)

improved_result_df.to_csv(RESULT_DIR / "improved_train_results.csv", index=False)

with open(RESULT_DIR / "improved_best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_improved, f, ensure_ascii=False, indent=2)

print("best improved")
print(json.dumps(best_improved, ensure_ascii=False, indent=2))

display(
    improved_result_df
    .sort_values("macro_f1", ascending=False)
    .head(10)[["theta", "zcr_min", "zcr_max", "median_width", "min_speech_frames", "Lh",
               "macro_f1", "f1_speech", "f1_non_speech", "TP", "FP", "FN", "TN"]]
)

## 4．A1の判定例

In [ ]:
example_id = "A1"
d = train_data[example_id]

params = {
    "theta": float(best_improved["theta"]),
    "zcr_min": float(best_improved["zcr_min"]),
    "zcr_max": float(best_improved["zcr_max"]),
    "median_width": int(best_improved["median_width"]),
    "min_speech_frames": int(best_improved["min_speech_frames"]),
    "Lh": int(best_improved["Lh"])
}

pred = predict_improved_from_features(
    {"S_band": d["S_band"], "zcr": d["zcr"]},
    **params
)

plt.figure(figsize=(14, 5))
plt.plot(d["times"], d["S_band"], label="speech-band relative energy [dB]")
plt.plot(d["times"], d["y"] * np.max(d["S_band"]), label="true label", alpha=0.7)
plt.plot(d["times"], pred * np.max(d["S_band"]), label="prediction", alpha=0.7)
plt.xlabel("Time [s]")
plt.ylabel("Score")
plt.title(f"Improved VAD example: {example_id}")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(14, 3))
plt.plot(d["times"], d["zcr"], label="ZCR")
plt.axhline(params["zcr_min"], linestyle="--", label="zcr_min")
plt.axhline(params["zcr_max"], linestyle="--", label="zcr_max")
plt.xlabel("Time [s]")
plt.ylabel("ZCR")
plt.title(f"Zero crossing rate: {example_id}")
plt.grid(True)
plt.legend()
plt.show()

display(pd.DataFrame([macro_f1_score(d["y"], pred)]))

## 5．改良内容の考察メモ

- 全帯域エネルギーでは，低周波雑音や環境音の影響を受けやすい．  
- 音声帯域エネルギーを使うことで，人の声に関係しやすい周波数成分を重視できる．  
- ZCRを使うことで，極端に変化が少ない無音区間や，高周波雑音が目立つ区間を抑制できる可能性がある．  
- 中央値フィルタ，短区間除去，hangoverにより，フレーム単位の細かな判定揺れを抑えられる．